<a href="https://colab.research.google.com/github/ep24b009-HariccharanM/Coding-Exercises/blob/main/Variational_Quantum_Eigensolver_2x2/VQE_2x2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install qiskit[visualization] qiskit-ibm-runtime qiskit-aer qiskit_qasm3_import

import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Pauli, SparsePauliOp, Statevector
from qiskit.visualization import plot_histogram, plot_bloch_multivector
from qiskit_aer import AerSimulator
from qiskit.circuit import Parameter, ParameterVector
import qiskit.qasm3
from qiskit_ibm_runtime.fake_provider import FakeVigoV2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler, EstimatorV2 as Estimator, QiskitRuntimeService

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.visualization import *
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit.primitives import StatevectorSampler, PrimitiveJob
import matplotlib.pyplot as plt
import math

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 6.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 88.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.8/386.8 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.5/102.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 541.5/541.5 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 92.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.0/218.0 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━

In [ ]:
from qiskit.quantum_info import Operator
import numpy as np

oper = np.array([[1,1j*3**0.5],[-1*1j*3**0.5,-1]])  #Arbitrary matrix
op = Operator(oper)

#The matrix is decomposed in to Pauli matrixes
a1 = float((oper[0,0] + oper[1,1])/2)
a2 = float((oper[0,1] + oper[1,0])/2)
a3 = float((oper[1,0] - oper[0,1])/2j)
a4 = float((oper[0,0] - oper[1,1])/2)

#Initializing the wavefunction with two parameters
ry = 0 #Rotation about Y
rz = 0 #Rotation about z

/tmp/ipykernel_515/136584630.py:8: ComplexWarning: Casting complex values to real discards the imaginary part
  a1 = float((oper[0,0] + oper[1,1])/2)
/tmp/ipykernel_515/136584630.py:9: ComplexWarning: Casting complex values to real discards the imaginary part
  a2 = float((oper[0,1] + oper[1,0])/2)
/tmp/ipykernel_515/136584630.py:10: ComplexWarning: Casting complex values to real discards the imaginary part
  a3 = float((oper[1,0] - oper[0,1])/2j)
/tmp/ipykernel_515/136584630.py:11: ComplexWarning: Casting complex values to real discards the imaginary part
  a4 = float((oper[0,0] - oper[1,1])/2)


In [ ]:
def anzx(ry,rz): #Measurement using Pauli X
  qc = QuantumCircuit(1,1)
  qc.ry(ry,0)
  qc.rz(rz,0)

  qc.h(0)
  qc.measure(0,0)

  sampler = StatevectorSampler()
  pub = (qc)
  job_sampler = sampler.run([pub], shots=10000)
  result_sampler = job_sampler.result()
  counts_sampler = result_sampler[0].data.c.get_counts()
  num_zeros = counts_sampler.get('0', 0)
  return num_zeros / 10000

In [ ]:
def anzy(ry,rz): #Measurement using Pauli Y
  qc = QuantumCircuit(1,1)
  qc.ry(ry,0)
  qc.rz(rz,0)

  qc.sdg(0)
  qc.h(0)
  qc.measure(0,0)

  sampler = StatevectorSampler()
  pub = (qc)
  job_sampler = sampler.run([pub], shots=10000)
  result_sampler = job_sampler.result()
  counts_sampler = result_sampler[0].data.c.get_counts()
  plot_histogram(counts_sampler)
  num_zeros = counts_sampler.get('0', 0)
  return num_zeros / 10000

In [ ]:
def anzz(ry,rz): #Measurement using Pauli Z
  qc = QuantumCircuit(1,1)
  qc.ry(ry,0)
  qc.rz(rz,0)

  qc.measure(0,0)

  sampler = StatevectorSampler()
  pub = (qc)
  job_sampler = sampler.run([pub], shots=10000)
  result_sampler = job_sampler.result()
  counts_sampler = result_sampler[0].data.c.get_counts()
  plot_histogram(counts_sampler)
  num_zeros = counts_sampler.get('0', 0)
  return num_zeros / 10000

In [ ]:
def f(ry,rz):
  y = a1 + a2*(2*anzx(ry,rz)-1) + a3*(2*anzy(ry,rz)-1) + a4*(2*anzz(ry,rz)-1)
  return y

def grad_f(ry, rz): #Changing the parameters using the gradient approach
    df_drx = (f(ry+0.1,rz) - f(ry-0.1,rz))/0.2
    df_dry = (f(ry,rz+0.1) - f(ry,rz-0.1))/0.2
    return df_drx, df_dry

# Initial guess
lr = 0.1
steps = 200

for i in range(steps):
    dry, drz = grad_f(ry, rz)
    ry -= lr * dry
    rz -= lr * drz
    c = f(ry, rz)
    print(f"Step {i+1}: ry={ry:.4f}, rz={rz:.4f}, c={c:.4f}")

print(f"Minimum at ry={ry:.4f}, rz={rz:.4f}, c={c:.4f}")

Step 1: ry=0.0024, rz=0.0087, c=1.0097
Step 2: ry=-0.0167, rz=-0.0068, c=0.9782
Step 3: ry=-0.0257, rz=-0.0030, c=0.9868
Step 4: ry=-0.0129, rz=-0.0035, c=1.0098
Step 5: ry=0.0007, rz=0.0001, c=0.9726
Step 6: ry=-0.0196, rz=0.0153, c=0.9900
Step 7: ry=-0.0301, rz=0.0018, c=1.0112
Step 8: ry=-0.0210, rz=-0.0015, c=0.9726
Step 9: ry=-0.0176, rz=-0.0063, c=1.0236
Step 10: ry=-0.0173, rz=0.0029, c=1.0031
Step 11: ry=-0.0138, rz=-0.0067, c=0.9595
Step 12: ry=-0.0142, rz=0.0025, c=1.0087
Step 13: ry=-0.0248, rz=-0.0133, c=0.9870
Step 14: ry=-0.0166, rz=-0.0260, c=1.0031
Step 15: ry=-0.0174, rz=-0.0291, c=0.9976
Step 16: ry=-0.0282, rz=-0.0372, c=0.9963
Step 17: ry=-0.0509, rz=-0.0338, c=0.9877
Step 18: ry=-0.0675, rz=-0.0392, c=1.0075
Step 19: ry=-0.0856, rz=-0.0518, c=1.0044
Step 20: ry=-0.1240, rz=-0.0459, c=1.0026
Step 21: ry=-0.1491, rz=-0.0580, c=0.9836
Step 22: ry=-0.1539, rz=-0.0872, c=0.9580
Step 23: ry=-0.1979, rz=-0.1262, c=0.9583
Step 24: ry=-0.2154, rz=-0.1594, c=0.8962
Step 25: 

In [ ]:
fin_ev = np.array([np.cos(ry/2), np.sin(ry/2)*np.exp(1j*rz)])
print("The eigen vector that has the least eigen value is\n",fin_ev)

The eigen vector that has the least eigen value is
 [ 0.49977592+0.j         -0.00592977+0.86613444j]
